# Signal-to-noise ratio criterion for strong lens detectability

This notebook demonstrates how signal-to-noise ratio is calculated in `SLSim` and how it can be used to filter out low SNR strong lenses when drawing a population.

In [ ]:
import os
import copy

import corner
import matplotlib.pyplot as plt
import numpy as np
import speclite

from astropy.cosmology import FlatLambdaCDM
from astropy.units import Quantity
from astropy.visualization import make_lupton_rgb

import slsim
import slsim.Pipelines as pipelines
import slsim.Sources as sources
import slsim.Deflectors as deflectors
from slsim.Lenses.lens_pop import LensPop
from slsim.ImageSimulation.image_simulation import simulate_image
from slsim.Pipelines import roman_speclite

In [2]:
# Import default Roman Space Telescope configuration
path = os.path.dirname(slsim.__file__)
module_path, _ = os.path.split(path)
skypy_config = os.path.join(module_path, "data/SkyPy/lsst-like_triple_SF.yml")


# generate Roman filters
roman_speclite.configure_roman_filters()


# import filter bands and make them recognizable in speclite
roman_filters = roman_speclite.filter_names()
# filters are ['Roman-F062', 'Roman-F087', 'Roman-F106', 'Roman-F129', 'Roman-F158', 'Roman-F184', 'Roman-F146', 'Roman-F213']


speclite.filters.load_filters(
    roman_filters[0],
    roman_filters[1],
    roman_filters[2],
    roman_filters[3],
    roman_filters[4],
    roman_filters[5],
    roman_filters[6],
    roman_filters[7],
)

## Set up a population of strong lenses

Here, we're simulating static, galaxy-galaxy strong lenses.

In [3]:
sky_area_value = 10
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
sky_area = Quantity(value=sky_area_value, unit="deg2")

band = "F106"

kwargs_deflector_cut = {"band": band, "band_max": 26.5, "z_min": 0.01, "z_max": 3}
kwargs_source_cut = {"band": band, "band_max": 26.5, "z_min": 0.01, "z_max": 6}

galaxy_simulation_pipeline = pipelines.SkyPyPipeline(
    skypy_config=skypy_config, sky_area=sky_area, filters=None, cosmo=cosmo
)

lens_galaxies = deflectors.EllipticalLensGalaxies(
    galaxy_list=galaxy_simulation_pipeline.red_galaxies,
    kwargs_cut=kwargs_deflector_cut,
    kwargs_mass2light={},
    cosmo=cosmo,
    sky_area=sky_area,
)

source_galaxies = sources.Galaxies(
    galaxy_list=galaxy_simulation_pipeline.blue_galaxies,
    kwargs_cut=kwargs_source_cut,
    cosmo=cosmo,
    sky_area=sky_area,
    catalog_type="skypy",
    source_size=None,
    extended_source_type="single_sersic", # "catalog_source" to use realistic sources, however opening files and loading images into memory is extremely slow
    #extended_source_kwargs={"catalog_type": "HST_COSMOS", "catalog_path": "/home/ahuang314/COSMOS_23.5_training_sample"}
)

gg_lens_pop = LensPop(
    deflector_population=lens_galaxies,
    source_population=source_galaxies,
    cosmo=cosmo,
    sky_area=sky_area,
)

/home/paras/repos/self/slsim/slsim/Deflectors/DeflectorPopulation/elliptical_lens_galaxies.py:48: UserWarning: Angular size is converted to arcsec because provided input_catalog_type is skypy. If this is not correct, please refer to the documentation of the class you are using
  galaxy_list = param_util.catalog_with_angular_size_in_arcsec(


In [8]:
snr_limit = {
    band: 20,
}

kwargs_lens_cuts = {
    "min_image_separation": 0.2,  # arcsec
    "mag_arc_limit": {band: 25},
    #"snr_limit": snr_limit,
}

sky_area_multiplier = 241

effective_sky_area = sky_area_value * sky_area_multiplier
print(f"Effective sky area: {effective_sky_area}")

Effective sky area: 2410


In [10]:
from tqdm import tqdm

gg_lens_population = []
    
for i in tqdm(range(sky_area_multiplier)):
    gg_lens_population = gg_lens_population + gg_lens_pop.draw_population(kwargs_lens_cuts=kwargs_lens_cuts, speed_factor=1000)

print("Number of lenses:", len(gg_lens_population))

100%|██████████| 241/241 [10:41<00:00,  2.66s/it]

Number of lenses: 48442


In [11]:
len(gg_lens_pop.draw_population(kwargs_lens_cuts=kwargs_lens_cuts, speed_factor=1))

1320

The following cell filters out the lenses with SNR > 20, which we use as an additional criterion for detectability

In [12]:
detectable = {"lenses": [], "snr": []}
exposure_time = 642

for gg_lens in gg_lens_population:
    snr = gg_lens.snr(
        band=band,
        fov_arcsec=10.01, 
        observatory="Roman",
        snr_per_pixel_threshold=1,
        exposure_time=exposure_time,
    )
    if snr is None:
        pass
    elif snr > 20:
        detectable['lenses'].append(gg_lens)
        detectable['snr'].append(snr)

detectable['snr'] = np.array(detectable['snr'])
print(f"{len(detectable['snr'])} detectable lenses")

TypeError: Lens.snr() got an unexpected keyword argument 'exposure_time'